# Task 1 — The Restricted Robot
### Testing: Span & Basis

## The scenario

An AI agent ("the robot") lives on an infinite 2D integer grid, starting at the
origin `(0, 0)`. It is **restricted**: it can only move using a fixed, small
set of "move vectors." Each move adds one of these vectors (or its negative —
the robot can step backward too) to its current position, any integer number
of times, in any order.

For example, if the robot's only allowed move vectors are `(1, 0)` and
`(0, 1)`, it can clearly reach *every* integer point `(x, y)` — that's just
normal grid movement.

But what if its move vectors are `(2, 0)` and `(0, 2)`? Or `(1, 1)` and
`(1, -1)`? Or `(2, 1)` and `(4, 2)`?

**The core question of this notebook:** given a set of move vectors, which
grid cells can the robot ever reach? This is *exactly* the question "what is
the span of these vectors?" — except here we care about the **integer**
combinations (a lattice), not just any real-number combination.

We will build every tool needed to answer this **from scratch** — no numpy,
no external linear algebra packages. Just plain Python + exact fraction
arithmetic where needed.

## Step 1 — Setting up

We import our own from-scratch linear algebra library (`src/linalg.py`).
Go peek at that file — every function in it (determinant, rank,
Gaussian elimination, extended Euclidean algorithm) is implemented with
plain Python lists and tuples.

In [ ]:
import sys
sys.path.insert(0, "../src")
from linalg import (
    columns_to_matrix, det, rank, is_linearly_independent, spans_r2,
    egcd, gcd, vec_add, scalar_mult
)
from fractions import Fraction
import matplotlib.pyplot as plt


## Step 2 — Span in the real-number sense

First, the *continuous* version of the question, ignoring integers for a
moment: do the move vectors span all of R^2 (the whole plane), or only a
line, or just the origin?

- If the two vectors are **linearly independent** (neither is a scalar
  multiple of the other), together they span all of R^2.
- If they're **parallel** (one is a scalar multiple of the other), they only
  span a line through the origin.
- If both vectors are `(0, 0)`, they span only the origin.

We check this with **rank**: build a matrix with the move vectors as
columns, and compute its rank via Gaussian elimination (`rref` in our
library).

In [ ]:
def describe_span(move_vectors):
    A = columns_to_matrix(move_vectors)
    r = rank(A)
    if r == 0:
        return "Span = just the origin. This robot cannot move at all."
    elif r == 1:
        return "Span = a single line through the origin (1-dimensional)."
    else:
        return "Span = all of R^2 (the move vectors are linearly independent)."

examples = [
    [(1, 0), (0, 1)],
    [(1, 1), (2, 2)],   # parallel -> span is a line
    [(0, 0), (0, 0)],   # no movement at all
    [(2, 1), (1, -1)],
]

for mv in examples:
    print(mv, "->", describe_span(mv))


## Step 3 — Basis: is the movement set "efficient"?

A set of vectors is a **basis** for R^2 if it (a) spans R^2 and (b) is
linearly independent (no redundant vectors). For exactly 2 vectors in R^2,
this just means: they're not parallel, i.e. `rank == 2`, i.e. their
determinant is **nonzero**.

If the move set is a basis, no move vector is "wasted" — every vector
contributes a genuinely new direction.

In [ ]:
def is_basis_r2(move_vectors):
    if len(move_vectors) != 2:
        return False  # for this exercise we only consider exactly 2 move vectors
    return is_linearly_independent(move_vectors)

for mv in examples:
    A = columns_to_matrix(mv)
    print(mv, "det =", det(A), "-> basis for R^2?", is_basis_r2(mv))


## Step 4 — The real question: the INTEGER lattice

Being a basis of R^2 is not the same as being able to reach *every grid
cell*. The robot can only take an **integer** number of steps of each move
vector (n1 and n2 both integers, possibly negative). The set of reachable
points is called the **lattice** generated by the move vectors:

```
L = { n1 * v1 + n2 * v2 : n1, n2 are integers }
```

Even if `v1, v2` form a basis of R^2 (determinant != 0), the lattice L might
still miss most integer points! Classic example: `v1 = (2, 0)`, `v2 = (0, 2)`.
These span all of R^2 and are linearly independent, but the robot can only
ever reach points where **both** coordinates are even.

**Key fact from linear algebra:** if `M` is the matrix with the move
vectors as columns, then the lattice L generated by the columns has exactly
`|det(M)|` points per unit area, compared to the full integer grid Z^2.
- `|det(M)| == 1` → the move vectors are a basis for the *lattice* Z^2 too.
  Every integer point is reachable.
- `|det(M)| > 1` → only 1 out of every `|det(M)|` grid cells is reachable.
- `det(M) == 0` → the reachable set collapses to a line (or just the
  origin), covered in Step 2 already.

In [ ]:
def lattice_index(move_vectors):
    """|det(M)| = how many grid cells there are per reachable point.
    Returns None if the move vectors don't even span R^2 (det == 0)."""
    A = columns_to_matrix(move_vectors)
    d = det(A)
    if d == 0:
        return None
    return abs(d)

for mv in examples:
    print(mv, "-> lattice index:", lattice_index(mv))


## Step 5 — Can the robot reach a *specific* target?

Now the practical question an intern would actually be asked: **given a
target cell `(tx, ty)`, can the robot get there using these move
vectors?**

This is a classic 2-variable **linear Diophantine system**:

```
n1 * v1x + n2 * v2x = tx
n1 * v1y + n2 * v2y = ty
```

We solve for `(n1, n2)` using **Cramer's rule** (exact, via `Fraction`, so
we never suffer floating point rounding), then check whether both `n1` and
`n2` come out to whole numbers.

In [ ]:
def can_reach(target, move_vectors):
    """Returns (reachable: bool, (n1, n2) or None)."""
    v1, v2 = move_vectors
    M = columns_to_matrix([v1, v2])
    D = det(M)
    if D == 0:
        # Degenerate: span is a line or the origin. Check if target lies on it.
        # Try to write target as a rational multiple of whichever vector is nonzero.
        for v in (v1, v2):
            if v != (0, 0):
                if v[0] != 0:
                    t = Fraction(target[0], v[0])
                elif v[1] != 0:
                    t = Fraction(target[1], v[1])
                else:
                    continue
                if vec_scale_matches(t, v, target):
                    return True, (t, 0) if v is v1 else (0, t)
        return (target == (0, 0)), None

    # Cramer's rule
    M1 = [[target[0], M[0][1]], [target[1], M[1][1]]]
    M2 = [[M[0][0], target[0]], [M[1][0], target[1]]]
    n1 = Fraction(det(M1), D)
    n2 = Fraction(det(M2), D)
    reachable = (n1.denominator == 1) and (n2.denominator == 1)
    return reachable, (n1, n2)


def vec_scale_matches(t, v, target):
    return (t * v[0], t * v[1]) == target


# --- Try it out ---
move_vectors = [(2, 1), (1, -1)]   # a basis for R^2; is it a basis for Z^2?
print("lattice index:", lattice_index(move_vectors))

targets = [(0, 0), (1, 0), (3, 0), (5, 2), (4, -2)]
for t in targets:
    reachable, steps = can_reach(t, move_vectors)
    print(f"target {t}: reachable = {reachable}, steps (n1, n2) = {steps}")


## Step 6 — Visualize the reachable region

Let's actually plot which grid cells in a window are reachable, for a couple
of interesting move-vector sets. This turns the abstract span/basis/lattice
math into something you can literally see.

In [ ]:
def plot_reachable(move_vectors, extent=6, title=None):
    reachable_pts, unreachable_pts = [], []
    for x in range(-extent, extent + 1):
        for y in range(-extent, extent + 1):
            ok, _ = can_reach((x, y), move_vectors)
            (reachable_pts if ok else unreachable_pts).append((x, y))

    fig, ax = plt.subplots(figsize=(5, 5))
    if unreachable_pts:
        ux, uy = zip(*unreachable_pts)
        ax.scatter(ux, uy, color="lightgray", s=25, label="unreachable")
    if reachable_pts:
        rx, ry = zip(*reachable_pts)
        ax.scatter(rx, ry, color="crimson", s=25, label="reachable")
    ax.scatter([0], [0], color="black", s=60, marker="*", label="start")
    ax.set_aspect("equal")
    ax.set_title(title or f"move vectors = {move_vectors}")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, linewidth=0.3)
    plt.show()

plot_reachable([(1, 0), (0, 1)], title="Full basis of Z^2: (1,0), (0,1) -- everything reachable")
plot_reachable([(2, 0), (0, 2)], title="Basis of R^2 but NOT of Z^2: only even cells")
plot_reachable([(1, 1), (1, -1)], title="Checkerboard lattice: (1,1), (1,-1)")
plot_reachable([(1, 2), (2, 4)], title="Parallel vectors: span is only a line")


## Step 7 — Exercises for interns

Try these on your own, using the functions above:

1. **Knight moves.** A robot restricted to chess-knight-style moves
   `(1, 2)` and `(2, 1)`. Is this a basis for Z^2? What's the lattice
   index? Can it reach `(1, 0)`? Can it reach `(1, 1)`?
2. **Three move vectors.** Extend `can_reach` to handle *three* move
   vectors instead of two (hint: with 3 vectors in 2D you have a choice
   of which two to solve with — or use integer combinations more
   generally via the GCD of all pairwise determinants).
3. **Minimum steps.** Among all valid `(n1, n2)` for a reachable point,
   is there always a way to compute the *minimum* number of total moves
   `|n1| + |n2|`? Think about when there could be multiple representations
   (hint: this only happens if you allow more than 2 move vectors).
4. **Real-world framing.** Explain in one paragraph, in your own words, why
   "span" answers the *continuous* reachability question, and why
   "determinant magnitude" answers the *discrete/lattice* reachability
   question. This distinction is exactly why a matrix can be invertible
   (det != 0) yet still not be **unimodular** (|det| == 1).